# Answer correctness (`LLM as a judge` method)

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import sys
sys.path.append("..")

import pandas as pd
from pathlib import Path

from evaluation import dataset, rag_metrics as ragm

SPLIT_PATH = Path("../data/eval/split.json")
dev, test = dataset.load_split(SPLIT_PATH)
print(f"{len(dev)} dev questions available")

20 dev questions available


## `Baseline` system with `Dense` retrieval and `k=5`

In [5]:
rows = ragm.evaluate_answers(dev, k=5)
df = pd.DataFrame(rows)
df[["id", "target_article", "article_hit", "verdict", "was_refused"]]

d:\Aanaconda\envs\ukask\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    loading embedding model, ~2GB, first run downloads it, be patient...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10859.74it/s]


    embedding model ready


,id,target_article,article_hit,verdict,was_refused
0,21,19,False,correct,False
1,26,21,True,partial,False
2,13,17,True,correct,False
3,64,40,True,correct,False
4,81,48,True,correct,False
5,48,2,False,correct,False
6,69,42,True,correct,False
7,3,29,True,correct,False
8,45,2,False,correct,False
9,92,6,False,correct,False


### Cross-tab

Rows are whether the labeled article got retrieved, columns are what the
judge said about the actual answer. The top-right cell (miss, but correct)
is exactly the pattern you noticed, and the bottom-left (hit, but wrong) is
worth knowing about too, right context, bad generation.

In [6]:
pd.crosstab(df["article_hit"], df["verdict"])

verdict,correct,incorrect,partial
article_hit,,,
False,7,0,0
True,10,1,2


### Disagreements

Especially the miss-but-correct cases

In [7]:
disagreements = df[
    ((df["article_hit"] == False) & (df["verdict"] == "correct")) |
    ((df["article_hit"] == True) & (df["verdict"] == "incorrect"))
]

for _, r in disagreements.iterrows():
    print(f"Q{r['id']}  article_hit={r['article_hit']}  verdict={r['verdict']}")
    print(f"  Q: {r['question']}")
    print(f"  ground truth: {r['ground_truth']}")
    print(f"  generated:    {r['generated_answer']}")
    print(f"  judge said:   {r['reasoning']}")
    print()

Q21  article_hit=False  verdict=correct
  Q: حداکثر مرخصی تحصیلی در کارشناسی پیوسته چند نیمسال است؟
  ground truth: دو نیمسال
  generated:    بر اساس تبصره 5 ماده 7 آیین‌نامه آموزشی، حداکثر مرخصی و حذف ترم مجاز (بدون احتساب در سنوات) برای دانشجویان دوره‌های کارشناسی پیوسته دو نیمسال می‌باشد. این محدودیت در سند 1، صفحه 24 و همچنین در سند 2 و 3 تحت عنوان تبصره 5 ماده 7 ذکر شده است.
  judge said:   پاسخ سیستم هم به لحاظ محتوایی صحیح است و حداکثر مرخصی تحصیلی را درست تعیین کرده است.

Q48  article_hit=False  verdict=correct
  Q: آیا تسهیلات حذف نمره مردودی شامل موارد تقلب می‌شود؟
  ground truth: خیر
  generated:    خیر، تسهیلات حذف نمره مردودی شامل دانشجویانی که به دلیل تقلب یا حکم انضباطی نمره مردودی دریافت می‌کنند، نمی‌شود (سند ۱، صفحه ۴، ماده ۲۰، تبصره ۲ و سند ۲، صفحه ۴، ماده ۱۱، تبصره ۲).
  judge said:   پاسخ سیستم دقیقاً با پاسخ صحیح مطابقت دارد.

Q45  article_hit=False  verdict=correct
  Q: عدم انتخاب واحد در یک نیمسال چه حکمی دارد؟
  ground truth: به عنوان انصراف از تحصیل تلقی می‌شود

### Saving the results

In [10]:
from evaluation import results

results.save_results(
    rows,
    "../data/eval/results/answers_baseline.json",
    label="baseline_dense_answers",
)